# Experiment 9: N-gram and Neural Language Models

A language model gives the probability of the next word.

- **Bigram (MLE):** P(w | prev) = count(prev, w) / count(prev). Unseen pairs get 0 -> infinite perplexity.
- **Laplace smoothing:** add 1 to every count.
- **Interpolation:** mix bigram and unigram: lambda * P_bigram + (1 - lambda) * P_unigram.
  (Kneser-Ney is a smarter version of this idea.)
- **LSTM:** neural network that reads the previous words and predicts the next one.
- **Perplexity** = exp(-average log probability). Lower is better.

In [ ]:
# !pip install nltk numpy tensorflow

In [ ]:
import math, random
from collections import Counter
import nltk
import numpy as np
import tensorflow as tf
from nltk.corpus import gutenberg

nltk.download('gutenberg', quiet=True)
random.seed(42); tf.keras.utils.set_random_seed(42)

## Data: Jane Austen's *Emma*, split 90/10

In [ ]:
sents = [[w.lower() for w in s if w.isalpha()] for s in gutenberg.sents('austen-emma.txt')]
sents = [s for s in sents if len(s) >= 3]
random.shuffle(sents)
cut = int(0.9 * len(sents))
train, test = sents[:cut], sents[cut:]

# rare words (seen once) become <unk>
counts = Counter(w for s in train for w in s)
vocab = {w for w, c in counts.items() if c >= 2} | {'<s>', '</s>', '<unk>'}

def prep(s):
    return ['<s>'] + [w if w in vocab else '<unk>' for w in s] + ['</s>']

train, test = [prep(s) for s in train], [prep(s) for s in test]
print('Train sentences:', len(train), '| Test:', len(test), '| Vocabulary:', len(vocab))

## Bigram models

In [ ]:
unigram = Counter(w for s in train for w in s)
bigram = Counter((s[i], s[i + 1]) for s in train for i in range(len(s) - 1))
total = sum(unigram.values())
V = len(vocab)

def p_mle(prev, w):
    return bigram[(prev, w)] / unigram[prev]

def p_laplace(prev, w):
    return (bigram[(prev, w)] + 1) / (unigram[prev] + V)

def p_interp(prev, w, lam=0.7):
    return lam * p_mle(prev, w) + (1 - lam) * unigram[w] / total

def perplexity(prob):
    log_sum, n = 0, 0
    for s in test:
        for prev, w in zip(s, s[1:]):
            p = prob(prev, w)
            if p == 0:
                return math.inf           # one unseen pair breaks MLE
            log_sum += math.log(p); n += 1
    return math.exp(-log_sum / n)

for name, fn in [('MLE', p_mle), ('Laplace', p_laplace), ('Interpolated', p_interp)]:
    print(f'{name:13} perplexity = {perplexity(fn):.1f}')

## Next-word prediction with the bigram model

In [ ]:
def next_words(prev, k=5):
    options = [(w, c) for (a, w), c in bigram.items() if a == prev]
    return sorted(options, key=lambda x: -x[1])[:k]

for word in ['she', 'mr', 'very']:
    print(word, '->', next_words(word))

## LSTM language model

In [ ]:
CONTEXT = 10                                   # how many previous words the LSTM sees
word2id = {w: i + 1 for i, w in enumerate(sorted(vocab))}   # 0 = padding
id2word = {i: w for w, i in word2id.items()}

def make_xy(data):
    X, y = [], []
    for s in data:
        ids = [word2id[w] for w in s]
        for i in range(1, len(ids)):
            ctx = ids[max(0, i - CONTEXT):i]
            X.append([0] * (CONTEXT - len(ctx)) + ctx)   # left-pad with 0
            y.append(ids[i])
    return np.array(X), np.array(y)

X_train, y_train = make_xy(train)
X_test, y_test = make_xy(test)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(CONTEXT,)),
    tf.keras.layers.Embedding(len(word2id) + 1, 100, mask_zero=True),
    tf.keras.layers.LSTM(128),
    tf.keras.layers.Dense(len(word2id) + 1, activation='softmax'),
])
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy')
model.fit(X_train, y_train, epochs=5, batch_size=256, validation_split=0.1, verbose=2)

# cross-entropy loss is mean -log P, so exp(loss) = perplexity
print('LSTM test perplexity:', round(math.exp(model.evaluate(X_test, y_test, verbose=0)), 1))

## Generate text

In [ ]:
def generate(seed, length=15):
    words = ['<s>'] + seed.split()
    for _ in range(length):
        ctx = [word2id.get(w, word2id['<unk>']) for w in words][-CONTEXT:]
        x = np.array([[0] * (CONTEXT - len(ctx)) + ctx])
        p = model.predict(x, verbose=0)[0]
        nxt = id2word.get(int(np.random.choice(len(p), p=p / p.sum())), '<unk>')   # sample
        if nxt == '</s>':
            break
        words.append(nxt)
    return ' '.join(words[1:])

for seed in ['emma was', 'she could not', 'mr elton']:
    print(generate(seed))